# Spectrum Objects

`specreduce` does not invent its own data class. Every operation in the package consumes and
returns a `specutils` **`Spectrum`**, and the previous chapter's `parse_image` exists to turn
whatever you hand in into one. Knowing what that container holds is the last piece of groundwork
before the reduction starts.

A `Spectrum` bundles four things that belong together:

| part | what it is | why it matters |
|---|---|---|
| `flux` | the data, as an `astropy` `Quantity` | carries its unit, so `electron` and `DN` cannot be confused |
| `spectral_axis` | one coordinate per sample | pixels before calibration, wavelength after |
| `uncertainty` | per-sample variance or standard deviation | drives extraction weights, clipping, detection thresholds |
| `mask` | boolean, `True` where data is bad | keeps dead pixels out of fits |

In [ ]:
import sys

sys.path.append("../src")

import astropy.units as u
import numpy as np
from astropy.nddata import (
    InverseVariance,
    StdDevUncertainty,
    VarianceUncertainty,
)
from specreduce.core import parse_image
from specutils import Spectrum

from common import read_data

arcs, lamps, obj = read_data()

## Building a `Spectrum` by hand

We start with the constructor, which takes the four pieces directly. Nothing here is specific to
`specreduce`. The five samples stand in for a short stretch of extracted spectrum in electrons,
and their uncertainty follows the CCD noise model.

In [ ]:
signal = np.array([12.0, 40.0, 95.0, 41.0, 11.0])
read_noise = 9.0

demo = Spectrum(
    flux=signal * u.electron,
    spectral_axis=np.arange(signal.size) * u.pix,
    uncertainty=StdDevUncertainty(np.sqrt(signal + read_noise**2)),
    mask=np.array([False, False, False, True, False]),
)

print(f"flux           {demo.flux}")
print(f"spectral_axis  {demo.spectral_axis}")
print(f"uncertainty    {demo.uncertainty.uncertainty_type}, {demo.uncertainty.array}")
print(f"mask           {demo.mask}")
print(f"shape          {demo.shape}")

Note the uncertainty is stored as a **standard deviation**, and the object records which
convention it uses in `uncertainty_type`; a variance or an inverse variance would be accepted as
well.

## A `Spectrum` is not necessarily one-dimensional

The name is misleading. `Spectrum` is the container for anything with a spectral axis, and that
includes the whole 2D frame. `parse_image`, the function every `specreduce` operation calls on its
input, returns a **2D** `Spectrum`:

In [ ]:
frame = parse_image(obj)

print(f"flux shape           {frame.flux.shape}")
print(f"spectral_axis shape  {frame.spectral_axis.shape}")
print(f"spectral_axis_index  {frame.spectral_axis_index}")
print(f"uncertainty shape    {frame.uncertainty.array.shape}")

`spectral_axis_index` is the attribute that makes sense of this: it says which array axis the
spectral axis describes. Here it is 1, matching Chapter 2's convention: dispersion on axis 1,
cross-dispersion on axis 0. The spectral axis itself has 1000 entries, one per dispersion column,
and is shared by all 372 rows.

So the same class covers the 2D frame going in and the 1D spectrum coming out. When reading
`specreduce` signatures, "image" and "spectrum" are both `Spectrum` objects; the difference is
dimensionality.

## Uncertainties: three conventions, one quantity

`astropy.nddata` expresses a per-sample uncertainty in one of three classes, `StdDevUncertainty`,
`VarianceUncertainty` and `InverseVariance`, which carry the same information in different forms.
`represent_as` converts between them, unit included:

In [ ]:
sigma = StdDevUncertainty([2.0, 3.0, 5.0], unit=u.electron)

for uncertainty in (
    sigma,
    sigma.represent_as(VarianceUncertainty),
    sigma.represent_as(InverseVariance),
):
    print(
        f"{uncertainty.uncertainty_type:>4}  {uncertainty.array}  [{uncertainty.unit}]"
    )

`specreduce` does not care which of the three it is given. Every operation converts its input to
the form it computes in, on a copy, and returns its result in the convention it received, so the
class is a choice of representation, not a constraint. The book's frames carry a
`StdDevUncertainty`, the convention the data files store, and every product in the book comes back
the same way. You meet `represent_as` only in code of your own, as when independent uncertainties
have to be added in quadrature and you need the variance.

## Units

Two independent unit systems live in a `Spectrum`: one on the flux, one on the spectral axis.

The **spectral axis** converts freely, because wavelength, frequency and energy are all the same
physical quantity viewed differently.

In [ ]:
calibrated_axis = Spectrum(
    flux=signal * u.electron,
    spectral_axis=(5000 + np.arange(signal.size) * 5.0) * u.AA,
    uncertainty=StdDevUncertainty(np.sqrt(signal + read_noise**2)),
)

print(f"wavelength  {calibrated_axis.spectral_axis}")
print(f"frequency   {calibrated_axis.spectral_axis.to(u.THz).round(3)}")
print(
    f"energy      {calibrated_axis.spectral_axis.to(u.eV, equivalencies=u.spectral()).round(4)}"
)

The **flux** unit does not convert so freely, and that is a feature:

In [ ]:
try:
    calibrated_axis.with_flux_unit(u.ct)
except u.UnitConversionError as error:
    print(f"UnitConversionError: {error}")

scaled = calibrated_axis.with_flux_unit(u.Unit(1e3 * u.electron))
print(f"\nrescaling the same unit is fine: {scaled.flux}")

`electron` and `ct` (count / ADU) are *not* convertible without knowing the detector gain, and
`astropy` refuses to guess. This is the units system doing its job: the frames were converted to electrons with `GAIN`
before the book starts (Chapter 1), precisely because photon statistics apply to detected charge,
not to digitized counts, and a container that let you silently relabel one as the other would make that distinction meaningless.

## The spectral axis and the WCS

The `spectral_axis` is the coordinate of each sample along the dispersion direction. It is a
`SpectralAxis`, a `Quantity` subclass, with one entry per sample: a pixel index before wavelength
calibration, a wavelength (or frequency, or energy) after. On a 2D frame there is still only one
axis, shared by every row.

The `wcs` holds the same information as a *transform* rather than a table of values: a mapping from
array index to world coordinate that can be evaluated anywhere, not only at the sample centers.
`Spectrum` accepts either. Given a `spectral_axis`, `specutils` builds a lookup-table WCS (a
`gwcs` object) from its values; given a `wcs`, it evaluates the transform at each pixel to fill in
the axis. The two stay consistent, so `spectral_axis` is always present and is the attribute to
read in ordinary code. The `wcs` is what a file format stores when a spectrum with a wavelength
solution is written out, and what a reader evaluates to recover the axis.

On a pixel axis the world coordinate has no physical meaning, and `astropy` will not build a
`SpectralCoord` from it, so before calibration only the `spectral_axis` is usable.

## Summary

- Every `specreduce` operation speaks `specutils` **`Spectrum`**: `flux` (a `Quantity`),
  `spectral_axis`, `uncertainty`, and `mask` in one object.
- **A `Spectrum` can be 2D.** `parse_image` returns the whole frame as one, with
  `spectral_axis_index` saying which axis is spectral (1, per Chapter 2's convention).
- Uncertainties come in three interchangeable conventions, and `represent_as` converts between
  them. `specreduce` accepts all three, computes in whichever it needs, and returns the one it was
  given; the book's data carry a `StdDevUncertainty`. An input with no uncertainty is given a
  variance of exactly one, so load through `common.read_data`.
- The spectral axis converts freely between wavelength, frequency and energy. The flux unit does
  not: `electron` and `ct` need the gain, and `astropy` will not guess.
- The `spectral_axis` is one coordinate per sample, a pixel index before calibration and a
  wavelength after. The `wcs` is the same mapping as a transform; `specutils` builds whichever
  one you did not supply, so `spectral_axis` is always there to read.

That completes the foundations. Next: [Tracing the Spectrum](04-tracing.ipynb), where the reduction
proper begins.

## Summary

- Every `specreduce` operation speaks `specutils` **`Spectrum`**: `flux` (a `Quantity`),
  `spectral_axis`, `uncertainty`, and `mask` in one object.
- **A `Spectrum` can be 2D.** `parse_image` returns the whole frame as one, with
  `spectral_axis_index` saying which axis is spectral (1, per Chapter 2's convention).
- Uncertainties come in three interchangeable conventions, and `represent_as` converts between
  them. `specreduce` accepts all three, computes in whichever it needs, and returns the one it was
  given; the book's data carry a `StdDevUncertainty`. An input with no uncertainty is given a
  variance of exactly one, so load through `common.read_data`.
- The spectral axis converts freely between wavelength, frequency and energy. The flux unit does
  not: `electron` and `ct` need the gain, and `astropy` will not guess.
- The `spectral_axis` is one coordinate per sample, a pixel index before calibration and a
  wavelength after. The `wcs` is the same mapping as a transform; `specutils` builds whichever
  one you did not supply, so `spectral_axis` is always there to read.

That completes the foundations. Next: [Tracing the Spectrum](04-tracing.ipynb), where the reduction
proper begins.